In [1]:
from __future__ import annotations

import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.distributions.normal import Normal

import gymnasium as gym


plt.rcParams["figure.figsize"] = (10, 5)

In [2]:
class AnimalType(enum.Enum):
    bee = 1
    ant = 2
    spider = 3
    grasshopper = 4
    
get_close_coords = lambda row, col: [
    (row + 1, col + 1),
    (row - 1, col + 1),
    (row + 2, col),
    (row - 2, col),
    (row + 1, col - 1),
    (row - 1, col - 1),
]  
    
class HiveGameState:
    def __init__(self, num_ants: int = 3, num_spiders=3, num_grasshoppers=3):
        self.num_ants = num_ants
        self.num_spiders = num_spiders
        self.num_grasshoppers = num_grasshoppers
        self.table = np.zeros((1000, 1000))
        self.player_table = np.zeros((1000, 1000))
        self.last_player_idx = 0
        self.turn_num = 0
        
        for animal_type, num in [
            (AnimalType.bee, 1), 
            (AnimalType.ant, self.num_ants),
            (AnimalType.spider, self.num_spiders), 
            (AnimalType.grasshopper, self.num_grasshoppers)
        ]:
            self.player_1_pieces += [(None, animal_type)] * num
            self.player_2_pieces += [(None, animal_type)] * num
        
    def apply_action(player_idx: Literal[1, 2], animal_idx, point_to) -> bool:
        if self.last_player_idx == player_idx:
            return False
        
        pieces = self.player_1_pieces if player_idx == 1 else self.player_2_pieces
        
        point_from, animal = pieces[animal_idx]
        
        if self.turn_num > 1 and animal != AnimalType.bee:
            return False
        
        if animal == AnimalType.ant:
            if not point_to in self._get_all_possible_dest_points_for_ant(point_from):
                return False
        elif animal == AnimalType.bee:
            if not point_to in self._get_all_possible_dest_points_for_bee(point_from):
                return False
        elif animal == AnimalType.spider:
            if not point_to in self._get_all_possible_dest_points_for_spider(point_from):
                return False
        elif animal == AnimalType.grasshopper:
            if not point_to in self._get_all_possible_dest_points_for_grasshopper(point_from):
                return False
        else:
            raise ValueError()
            
        pieces[animal_idx] = (point_to, animal)
        if point_from is not None:
            self.table[point_from[0], point_from[1]] = 0
            self.player_table[point_from[0], point_from[1]] = 0
        self.table[point_to[0], point_to[1]] = animal.value
        self.player_table[point_to[0], point_to[1]] = player_idx
        self.last_player_idx = player
        self.turn_num += 1
        
    @staticmethod
    def _get_allocation_points(table, player_idx, player_table):
        if self.turn_num == 0:
            return [(500, 500)]
        if self.turn_num == 1:
            (row, col) = np.where(table != 0)[0]
            return get_close_coords(row, col)
        res = {}
        enemy_table = ~((player_table == 0) | (player_table == player_idx))
        potential_points = []
        for (allocated_point), animal in pieces:
            row, col = allocated_point
            potential_points.append(np.array(get_close_coords(row, col)))
        potential_points = np.unique(np.concatenate(potential_points))
        is_close_to_enemy = (
            enemy_table[(potential_points[:, 0] - 1, potential_points[:, 1] - 1)]
            |
            enemy_table[(potential_points[:, 0] - 1, potential_points[:, 1] + 1)]
            | 
            enemy_table[(potential_points[:, 0] + 1, potential_points[:, 1] - 1)]
            |
            enemy_table[(potential_points[:, 0] + 1, potential_points[:, 1] + 1)]
            | 
            enemy_table[(potential_points[:, 0] + 2, potential_points[:, 1])]
            |
            enemy_table[(potential_points[:, 0] - 2, potential_points[:, 1])]
        )
        potential_points = potential_points[~is_close_to_enemy]
        return potential_points
        
    def _walk(start, table, rule):
        q = queue.Queue()
        q.push(start[0], start[1])
        visited_table = np.zeros(table.shape)

        while len(q):
            x, y = q.pop()
            visited_table[x, y] = 1
            next_point_seq = rule(x, y)
            for next_point in next_point_seq:
                if not visited_table[next_point]:
                    q.push(next_point)
        visited_table[start[0], start[1]] = 0
        x_seq, y_seq = np.where(visited_table != 0)
        res = np.vstack((x_seq, y_seq)).T
        return res
    
    @staticmethod
    def _is_movement_locked(point, point_from, table):
        if point[0] == point_from[0] + 1 and point[1] == point_from[1] + 1:
            if table[point[0], point[1] + 2] != 0 and table[point[0] + 1, point[1] - 1] != 0:
                return True
        if point[0] == point_from[0] - 1 and point[1] == point_from[1] + 1:
            if table[point[0], point[1] + 2] != 0 and table[point[0] - 1, point[1] - 1] != 0:
                return True
        if point[0] == point_from[0] - 1 and point[1] == point_from[1] - 1:
            if table[point[0], point[1] - 2] != 0 and table[point[0] - 1, point[1] + 1] != 0:
                return True
        if point[0] == point_from[0] + 1 and point[1] == point_from[1] - 1:
            if table[point[0], point[1] - 2] != 0 and table[point[0] + 1, point[1] + 1] != 0:
                return True
        if point[0] == point_from[0] and point[1] == point_from[1] + 2:
            if table[point[0] - 1, point[1] + 1] != 0 and table[point[0] + 1, point[1] + 1] != 0:
                return True
        if point[0] == point_from[0] and point[1] == point_from[1] - 2:
            if table[point[0] - 1, point[1] - 1] != 0 and table[point[0] + 1, point[1] - 1] != 0:
                return True
        return False
    
    def _get_all_possible_dest_points_for_bee(self, player_idx, point_from):
        if point_from is None:
            return self._get_allocation_points(self.table, player_idx, self.player_table)
        
        if _is_graph_component_more_than_1(remove_point_from_table(self.table, point_from))
            return []
            
        next_points = np.array(get_close_coords(point_from))
        res = []
        for point_to in next_points:
            if table[point_to[0], point_to[1]] != 0:
                # next point is not free
                continue
            next_to_point_to = np.array(get_close_coords(point_to))
            if (table[next_to_point_to[:, 0], next_to_point_to[:, 1]] != 0).sum() == 0:
                # jump to void, no neighbours
                continue
            # next_point is outside of hive
            if _is_movement_locked(point_to, point_from, self.table):
                continue
            # next_points don't cause gap in hive

            res.append(point)
        return res
        
    def _get_all_possible_dest_points_for_ant(self, point_from):    
        if point_from is None:
            return self._get_allocation_points(self.table, player_idx, self.player_table)
                next_points = np.array(get_close_coords(point_from))
        
        for pass in np.array(get_close_coords(point_from)):
        next_points = np.array(get_close_coords(point_from))
        res = []
        for point_to in next_points:
            if table[point_to[0], point_to[1]] != 0:
                # next point is not free
                continue
            next_to_point_to = np.array(get_close_coords(point_to))
            if (table[next_to_point_to[:, 0], next_to_point_to[:, 1]] != 0).sum() == 0:
                # jump to void, no neighbours
                continue
            # next_point is outside of hive
            if _is_movement_locked(point_to, point_from, self.table):
                continue
            # next_points don't cause gap in hive
            if _is_graph_component_more_than_1(remove_point_from_table(self.table, point_from))
                continue
            res.append(point)
        return res
        
    def _get_all_possible_dest_points_for_spider(self, point_from):
        if point_from is None:
            return self._get_allocation_points(self.table, player_idx, self.player_table)
        
    
    def _get_all_possible_dest_points_for_grasshopper(self, point_from):
        if point_from is None:
            return self._get_allocation_points(self.table, player_idx, self.player_table)

    def who_win(self) -> Optional[int]:
        pass
    
    
    def get_state(self):
        pass


NameError: name 'enum' is not defined